# Stage 4 — Build the Supervised Dataset

Turn three gridded cubes (CHL, SST, bathy) into a tidy table of `(features at time t) → (CHL at t+h)`, keeping gaps as `NaN` and respecting the no-shuffling-time rule.

## Why this is the riskiest stage in the whole project
Most silent bugs in time-series ML are born here:
- **Off-by-one shifts** between features and targets → leakage → impressive-looking R² that is pure noise.
- **Mixed lag direction** (lags going forward, or targets going backward) → same outcome.
- **Coord misalignment** between predictors and target → features paired with the wrong pixel.
- **Accidentally filling NaNs** when constructing the target → invalidates the methodological core.

The validation gate at the end runs a per-pixel **leakage assert** that catches all four. **Do not skip the gate.** PLAN.md: *"The single most important check in the whole project is the leakage assert."*

## What this notebook produces
- A parquet file `data/processed/supervised_sample.parquet`
- One row per (pixel × time) sampled to keep memory tractable
- Columns: `time, latitude, longitude, <features…>, target_h1 … target_h7`
- Gaps preserved as NaN — XGBoost will use them natively in Stage 6

## 4.0 — Configuration

Read `coverage_decision.json` from Notebook 2. A `DAILY` decision uses daily CHL at `t+h`. If the decision is `3-DAY COMPOSITE`, only the target changes: take the forward linear-space mean over `t+h`, `t+h+1`, and `t+h+2`, require at least two observations, and apply `log10` after averaging. Issue-time CHL features always remain daily.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xarray as xr

RAW_DIR = Path("data/raw")
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

CHL_PATH = RAW_DIR / "chl_malta_2021_2023.nc"
SST_PATH = RAW_DIR / "sst_malta_2021_2023.nc"
BATHY_PATH = RAW_DIR / "bathy_malta.nc"

DECISION_PATH = PROC_DIR / "coverage_decision.json"
OUT_PATH = PROC_DIR / "supervised_sample.parquet"
MANIFEST_PATH = PROC_DIR / "feature_manifest.json"

for path in (CHL_PATH, SST_PATH, BATHY_PATH, DECISION_PATH):
    assert path.exists(), f"Missing {path} — run the earlier notebooks first."

HORIZONS = [1, 2, 3, 4, 5, 6, 7]
LAGS = [1, 2, 3, 7, 14]
ROLL_WIN = 7

TARGET_COLS = [f"target_h{h}" for h in HORIZONS]
META_COLS = ["time", "latitude", "longitude", "month"]

# 10,000 rows from each individual year-month:
# 36 months × 10,000 = 360,000 rows.
SAMPLE_PER_YEAR_MONTH = 10_000
SEED = 42

decision_meta = json.loads(DECISION_PATH.read_text())
TARGET_MODE = str(decision_meta["decision"]).strip().upper()

allowed_modes = {"DAILY", "3-DAY COMPOSITE"}
if TARGET_MODE not in allowed_modes:
    raise RuntimeError(
        f"Unsupported coverage decision {TARGET_MODE!r}; "
        f"expected one of {sorted(allowed_modes)}"
    )

USE_3DAY_TARGET = TARGET_MODE == "3-DAY COMPOSITE"

print(f"Coverage decision : {TARGET_MODE}")
print(f"Horizons          : {HORIZONS}")
print(f"Lags              : {LAGS}")
print(f"Rolling window    : {ROLL_WIN}")
print(f"Sample/year-month : {SAMPLE_PER_YEAR_MONTH:,}")

Coverage decision : DAILY
Horizons          : [1, 2, 3, 4, 5, 6, 7]
Lags              : [1, 2, 3, 7, 14]
Rolling window    : 7
Sample/year-month : 10,000


## 4.1 — Load cubes & align onto the CHL grid

**Rule:** the CHL grid is finest, so everything gets regridded *onto it*, never the other way around. Spatial interpolation of SST (~5 km → 1 km) does not invent real detail — it just makes the arrays addressable on a common grid. For a tree model this is fine; coarse predictors contribute large-scale signal, which is exactly what we want from them.

In [2]:
chl_ds = xr.open_dataset(CHL_PATH)
sst_ds = xr.open_dataset(SST_PATH)
bathy_ds = xr.open_dataset(BATHY_PATH)

# Reapply the provider validity limits before taking log10.
chl_raw = chl_ds["CHL"]
chl_valid_min = float(chl_raw.attrs.get("valid_min", 0.0))
chl_valid_max = float(chl_raw.attrs.get("valid_max", np.inf))

chl = chl_raw.where(
    (chl_raw > 0)
    & (chl_raw >= chl_valid_min)
    & (chl_raw <= chl_valid_max)
)

log_chl = np.log10(chl).rename("log_chl")

# Unit-aware SST conversion.
sst = sst_ds["analysed_sst"]
sst_units = str(sst.attrs.get("units", "")).strip().lower()

if sst_units in {"kelvin", "k"}:
    sst = sst - 273.15
    sst.attrs["units"] = "degC"
elif sst_units not in {
    "celsius",
    "degree_celsius",
    "degrees_celsius",
    "degc",
    "°c",
}:
    raise RuntimeError(f"Unrecognised SST units: {sst_units!r}")

# Notebook 3 established exact temporal equality. Do not use nearest-time
# matching here because it could silently select the wrong day.
if not np.array_equal(sst.time.values, chl.time.values):
    raise RuntimeError("CHL and SST time coordinates are not identical.")

sst_on_chl = sst.interp(
    latitude=chl.latitude,
    longitude=chl.longitude,
)

# Normalise possible GEBCO coordinate names.
elev_var = next(
    v for v in bathy_ds.data_vars
    if v in ("z", "elevation", "bed_elevation")
)
elev = bathy_ds[elev_var]

rename_coords = {}
if "lat" in elev.dims:
    rename_coords["lat"] = "latitude"
if "lon" in elev.dims:
    rename_coords["lon"] = "longitude"
if rename_coords:
    elev = elev.rename(rename_coords)

elev = elev.sortby(["latitude", "longitude"])

# Interpolate signed elevation first, then convert ocean elevation to
# positive water depth. Land remains zero.
elev_on_chl = elev.interp(
    latitude=chl.latitude,
    longitude=chl.longitude,
)
depth_on_chl = xr.where(
    elev_on_chl < 0,
    -elev_on_chl,
    0,
).rename("depth_m")

print(f"CHL   : {dict(chl.sizes)}")
print(f"SST   : {dict(sst_on_chl.sizes)} — °C, regridded")
print(f"Depth : {dict(depth_on_chl.sizes)} — positive metres")

CHL   : {'time': 1095, 'latitude': 198, 'longitude': 272}
SST   : {'time': 1095, 'latitude': 198, 'longitude': 272} — °C, regridded
Depth : {'latitude': 198, 'longitude': 272} — positive metres


## 4.2 — Define the target base

Build `target_base` from the coverage decision. Daily mode uses daily `log10(CHL)`. Composite mode constructs the future three-day mean in linear CHL and then transforms it. `log_chl_t0` and every CHL lag are always derived from the daily issue-time series, preventing future target information from entering the features.


In [3]:
# Features always use daily CHL available at or before issue time t.
# Only the forecast target changes if the composite fallback was selected.
if not USE_3DAY_TARGET:
    target_base = log_chl.rename("target_base")
    print("Target: daily log10(CHL)")
else:
    # At time u, form the forward window CHL[u], CHL[u+1], CHL[u+2].
    # Composite in linear CHL, require at least two observations, and
    # apply log10 only after compositing.
    forward_chl = xr.concat(
        [chl.shift(time=-offset) for offset in range(3)],
        dim=xr.IndexVariable("composite_day", [0, 1, 2]),
    )

    forward_count = forward_chl.count("composite_day")
    composite_linear = (
        forward_chl
        .mean("composite_day", skipna=True)
        .where(forward_count >= 2)
    )

    target_base = np.log10(composite_linear).rename("target_base")

    print(
        "Target: log10(mean linear CHL over t+h through t+h+2), "
        "minimum two observed days"
    )

Target: daily log10(CHL)


## 4.3 — Build features (lagged, rolling, gap-age, SST, static, temporal)

Five feature families:
1. **Autoregressive lags** — `log_chl` at t-1, t-2, t-3, t-7, t-14.
2. **Rolling statistics** — 7-day mean & std of `log_chl`.
3. **Gap-age** — days since last valid CHL observation, per pixel. Encodes *how stale* the last observation is.
4. **SST features** — t0, t-1, and 7-day rolling mean.
5. **Static + temporal** — bathymetric depth, day-of-year `sin`/`cos`.

All lags go **backward** (`shift(time=+lag)`); targets go **forward** (`shift(time=-h)`). Mixing those directions is the classic catastrophic bug — the gate catches it.

In [4]:
import warnings

feat = xr.Dataset()

# Current and historical CHL features remain daily in both target modes.
feat["log_chl_t0"] = log_chl

for lag in LAGS:
    feat[f"log_chl_lag{lag}"] = log_chl.shift(time=lag)

feat["log_chl_roll7_mean"] = (
    log_chl.rolling(time=ROLL_WIN, min_periods=1).mean()
)

# NumPy can emit an expected warning for entirely empty rolling windows.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="Degrees of freedom <= 0 for slice",
    )
    feat["log_chl_roll7_std"] = (
        log_chl.rolling(time=ROLL_WIN, min_periods=2).std()
    )

feat["sst_t0"] = sst_on_chl
feat["sst_lag1"] = sst_on_chl.shift(time=1)
feat["sst_roll7_mean"] = (
    sst_on_chl.rolling(time=ROLL_WIN, min_periods=1).mean()
)

feat["depth_m"] = depth_on_chl.broadcast_like(log_chl)

doy = log_chl.time.dt.dayofyear
feat["doy_sin"] = (
    np.sin(2 * np.pi * doy / 365.25).broadcast_like(log_chl)
)
feat["doy_cos"] = (
    np.cos(2 * np.pi * doy / 365.25).broadcast_like(log_chl)
)

print("Feature variables:")
for name in feat.data_vars:
    print(f"  {name}")

Feature variables:
  log_chl_t0
  log_chl_lag1
  log_chl_lag2
  log_chl_lag3
  log_chl_lag7
  log_chl_lag14
  log_chl_roll7_mean
  log_chl_roll7_std
  sst_t0
  sst_lag1
  sst_roll7_mean
  depth_m
  doy_sin
  doy_cos


## 4.4 — Gap-age feature

Per pixel, days since the last valid CHL observation. Implemented as a vectorised loop over the time axis on the underlying NumPy array — pure xarray `.shift` chains would explode in memory.

In [5]:
observed_now = np.isfinite(log_chl.values)

# Float is required because "no observation has ever been seen yet"
# must be represented by NaN, not zero.
gap_age = np.full(observed_now.shape, np.nan, dtype=np.float32)
last_seen = np.full(observed_now.shape[1:], -1, dtype=np.int32)

for t in range(observed_now.shape[0]):
    observed_t = observed_now[t]

    last_seen[observed_t] = t
    seen_before_or_now = last_seen >= 0

    gap_age[t, seen_before_or_now] = (
        t - last_seen[seen_before_or_now]
    )

feat["gap_age"] = xr.DataArray(
    gap_age,
    dims=log_chl.dims,
    coords=log_chl.coords,
    name="gap_age",
    attrs={
        "long_name": "Days since most recent valid CHL observation",
        "definition": (
            "0 where CHL is valid; positive after a previously valid "
            "observation; NaN before the first valid observation"
        ),
    },
)

historical_gaps = gap_age[
    (~observed_now) & np.isfinite(gap_age)
]

print(
    "Maximum gap age after an observation:",
    float(historical_gaps.max()) if historical_gaps.size else np.nan,
)
print(
    "Cells occurring before their pixel's first observation:",
    int(np.isnan(gap_age).sum()),
)

Maximum gap age after an observation: 1037.0
Cells occurring before their pixel's first observation: 8549982


## 4.5 — Forecast targets

One target per horizon. `shift(time=-h)` moves observations from `t+h` back to row `t`, so `target_h{h}` at row `t` literally is *the value at* `t+h`. That's the contract the gate will verify.

In [6]:
targets = xr.Dataset()

for h in HORIZONS:
    # At issue time t, this selects target_base at t+h.
    targets[f"target_h{h}"] = target_base.shift(time=-h)

print("Target variables:", list(targets.data_vars))

Target variables: ['target_h1', 'target_h2', 'target_h3', 'target_h4', 'target_h5', 'target_h6', 'target_h7']


## 4.6 — Keep the full cube gridded

Merge features and targets with exact coordinates, but do not flatten the complete approximately 59-million-row rectangle. Sampling directly from the gridded arrays avoids several gigabytes of unnecessary pandas materialisation.


In [7]:
# Keep the data gridded here. Flattening the complete 58.9-million-row
# rectangle wastes several gigabytes and is unnecessary.
combined = xr.merge([feat, targets], join="exact")

rectangular_rows = (
    combined.sizes["time"]
    * combined.sizes["latitude"]
    * combined.sizes["longitude"]
)

print(f"Combined cube dimensions: {dict(combined.sizes)}")
print(f"Potential rectangular rows: {rectangular_rows:,}")
print("The cube will be sampled before conversion to pandas.")

Combined cube dimensions: {'latitude': 198, 'longitude': 272, 'time': 1095}
Potential rectangular rows: 58,972,320
The cube will be sampled before conversion to pandas.


## 4.7 — Sample by year-month

Draw up to 10,000 unique rows from each of the 36 year-month strata with a fixed seed. A row is eligible when at least one forecast target is observed. Feature NaNs—including missing current CHL—are deliberately retained so XGBoost can learn and be evaluated under cloudy issue-time conditions. In the validated run every stratum supplies the full quota, producing exactly 360,000 rows.


In [8]:
rng = np.random.default_rng(SEED)

time_index = pd.DatetimeIndex(combined.time.values)
time_periods = time_index.to_period("M")

expected_periods = pd.period_range(
    time_index.min().to_period("M"),
    time_index.max().to_period("M"),
    freq="M",
)

selected_time = []
selected_lat = []
selected_lon = []

for period in expected_periods:
    time_positions = np.flatnonzero(time_periods == period)

    # A row is useful if at least one forecast horizon has an observed
    # target. Feature NaNs, including missing current CHL, are retained.
    target_available = (
        targets[TARGET_COLS]
        .isel(time=time_positions)
        .to_array("target")
        .notnull()
        .any("target")
        .values
    )

    available_flat = np.flatnonzero(target_available)

    if available_flat.size == 0:
        raise RuntimeError(f"No usable target rows in {period}")

    n_take = min(SAMPLE_PER_YEAR_MONTH, available_flat.size)

    chosen_flat = rng.choice(
        available_flat,
        size=n_take,
        replace=False,
    )

    local_t, lat_i, lon_i = np.unravel_index(
        chosen_flat,
        target_available.shape,
    )

    selected_time.extend(time_positions[local_t].tolist())
    selected_lat.extend(lat_i.tolist())
    selected_lon.extend(lon_i.tolist())

selected = combined.isel(
    time=xr.DataArray(
        np.asarray(selected_time),
        dims="row",
    ),
    latitude=xr.DataArray(
        np.asarray(selected_lat),
        dims="row",
    ),
    longitude=xr.DataArray(
        np.asarray(selected_lon),
        dims="row",
    ),
)

sample = (
    selected
    .to_dataframe()
    .reset_index(drop=True)
)

sample["month"] = sample["time"].dt.month

sample = (
    sample
    .sort_values(["time", "latitude", "longitude"])
    .reset_index(drop=True)
)

print(f"Sample shape: {sample.shape}")
print("\nRows per year-month:")
print(sample.groupby(sample["time"].dt.to_period("M")).size())

print("\nSample NaN fractions:")
print((sample.isna().mean() * 100).round(1).rename("%NaN"))

Sample shape: (360000, 26)

Rows per year-month:
time
2021-01    10000
2021-02    10000
2021-03    10000
2021-04    10000
2021-05    10000
2021-06    10000
2021-07    10000
2021-08    10000
2021-09    10000
2021-10    10000
2021-11    10000
2021-12    10000
2022-01    10000
2022-02    10000
2022-03    10000
2022-04    10000
2022-05    10000
2022-06    10000
2022-07    10000
2022-08    10000
2022-09    10000
2022-10    10000
2022-11    10000
2022-12    10000
2023-01    10000
2023-02    10000
2023-03    10000
2023-04    10000
2023-05    10000
2023-06    10000
2023-07    10000
2023-08    10000
2023-09    10000
2023-10    10000
2023-11    10000
2023-12    10000
Freq: M, dtype: int64

Sample NaN fractions:
latitude               0.0
longitude              0.0
time                   0.0
log_chl_t0            42.4
log_chl_lag1          42.9
log_chl_lag2          42.6
log_chl_lag3          42.8
log_chl_lag7          43.4
log_chl_lag14         43.4
log_chl_roll7_mean     2.4
log_chl_roll7_std  

---
## Validation Gate 4 — Alignment, leakage, and sampling

The gate verifies that each `target_h{h}` equals `target_base` at `t+h`, lag 1 points strictly to daily CHL at `t-1`, and `log_chl_t0` contains only daily issue-time observations. It also checks the exact meaning of `gap_age`, all 36 year-month quotas, unique sample keys, retention of missing current CHL, and usable observations in every target column.

Any failure indicates an alignment, leakage, or sampling-contract error and must be fixed before baselines are computed.


In [9]:
results = {}

# Select three pixels with the greatest number of valid daily CHL values.
pixel_counts = log_chl.notnull().sum("time").values
top_flat = np.argsort(pixel_counts.ravel())[-3:]
test_lat_i, test_lon_i = np.unravel_index(
    top_flat,
    pixel_counts.shape,
)

# 1. Forecast target alignment
target_alignment_ok = True

for lat_i, lon_i in zip(test_lat_i, test_lon_i):
    for h in HORIZONS:
        lhs = (
            targets[f"target_h{h}"]
            .isel(
                latitude=lat_i,
                longitude=lon_i,
                time=slice(None, -h),
            )
            .values
        )

        rhs = (
            target_base
            .isel(
                latitude=lat_i,
                longitude=lon_i,
                time=slice(h, None),
            )
            .values
        )

        if not np.allclose(lhs, rhs, equal_nan=True):
            target_alignment_ok = False

results["targets equal target_base at t+h"] = target_alignment_ok

# 2. Lag direction
lag_alignment_ok = True

for lat_i, lon_i in zip(test_lat_i, test_lon_i):
    lhs = (
        feat["log_chl_lag1"]
        .isel(
            latitude=lat_i,
            longitude=lon_i,
            time=slice(1, None),
        )
        .values
    )

    rhs = (
        log_chl
        .isel(
            latitude=lat_i,
            longitude=lon_i,
            time=slice(None, -1),
        )
        .values
    )

    if not np.allclose(lhs, rhs, equal_nan=True):
        lag_alignment_ok = False

results["lag1[t] equals daily CHL[t-1]"] = lag_alignment_ok

# 3. Current CHL remains daily rather than a future composite
current_feature_ok = True

for lat_i, lon_i in zip(test_lat_i, test_lon_i):
    lhs = feat["log_chl_t0"].isel(
        latitude=lat_i,
        longitude=lon_i,
    ).values
    rhs = log_chl.isel(
        latitude=lat_i,
        longitude=lon_i,
    ).values

    if not np.allclose(lhs, rhs, equal_nan=True):
        current_feature_ok = False

results["log_chl_t0 contains only daily issue-time CHL"] = (
    current_feature_ok
)

# 4. Gap-age meaning
gap_values = feat["gap_age"].values

gap_zero = (
    np.isfinite(gap_values)
    & (gap_values == 0)
)

results["gap_age is zero exactly where CHL is observed"] = (
    np.array_equal(gap_zero, observed_now)
)

results["gap_age is never negative"] = (
    float(np.nanmin(gap_values)) >= 0
)

# 5. Sampling contracts
sample_periods = set(
    sample["time"].dt.to_period("M").unique()
)
required_periods = set(expected_periods)

results["all 36 year-month strata are present"] = (
    sample_periods == required_periods
)

rows_per_period = sample.groupby(
    sample["time"].dt.to_period("M")
).size()

results["each year-month has 10,000 rows"] = bool(
    (rows_per_period == SAMPLE_PER_YEAR_MONTH).all()
)

results["sample keys are unique"] = not sample.duplicated(
    ["time", "latitude", "longitude"]
).any()

# Missing feature values must be retained so the model learns cloudy cases.
results["sample includes missing current CHL"] = (
    sample["log_chl_t0"].isna().any()
)

# Every sampled row must be useful for at least one horizon.
results["no row has every target missing"] = not (
    sample[TARGET_COLS].isna().all(axis=1).any()
)

results["every target column has observations"] = all(
    sample[col].notna().any()
    for col in TARGET_COLS
)

for name, ok in results.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

print()
print(f"Sample rows                  : {len(sample):,}")
print(
    "Missing current CHL         : "
    f"{sample['log_chl_t0'].isna().mean() * 100:.1f}%"
)
print(
    "Missing lag-1 CHL           : "
    f"{sample['log_chl_lag1'].isna().mean() * 100:.1f}%"
)
print(
    "Missing H+1 target          : "
    f"{sample['target_h1'].isna().mean() * 100:.1f}%"
)
print(
    "Missing H+7 target          : "
    f"{sample['target_h7'].isna().mean() * 100:.1f}%"
)

failed = [name for name, ok in results.items() if not ok]

if failed:
    raise RuntimeError(
        "Validation Gate 4 failed:\n- "
        + "\n- ".join(failed)
    )

sample.to_parquet(OUT_PATH, index=False)

print("\nGate 4 PASSED — no leakage detected.")
print(
    f"Saved → {OUT_PATH} "
    f"({OUT_PATH.stat().st_size / 1e6:.1f} MB)"
)

PASS  targets equal target_base at t+h
PASS  lag1[t] equals daily CHL[t-1]
PASS  log_chl_t0 contains only daily issue-time CHL
PASS  gap_age is zero exactly where CHL is observed
PASS  gap_age is never negative
PASS  all 36 year-month strata are present
PASS  each year-month has 10,000 rows
PASS  sample keys are unique
PASS  sample includes missing current CHL
PASS  no row has every target missing
PASS  every target column has observations

Sample rows                  : 360,000
Missing current CHL         : 42.4%
Missing lag-1 CHL           : 42.9%
Missing H+1 target          : 41.6%
Missing H+7 target          : 41.9%

Gate 4 PASSED — no leakage detected.
Saved → data\processed\supervised_sample.parquet (31.7 MB)


## 4.8 — Persist the feature contract

Save the ordered feature, target, and metadata columns together with the horizon list, lag/rolling settings, target mode, coverage-decision source, sampling policy, quota, and realized row count. Downstream notebooks read this manifest instead of reconstructing the schema.


In [10]:
FEATURE_COLS = [
    col
    for col in sample.columns
    if col not in META_COLS + TARGET_COLS
]

manifest = {
    "meta_cols": META_COLS,
    "target_cols": TARGET_COLS,
    "feature_cols": FEATURE_COLS,
    "horizons": HORIZONS,
    "lags": LAGS,
    "roll_window": ROLL_WIN,
    "target_mode": TARGET_MODE,
    "use_3day_target": USE_3DAY_TARGET,
    "coverage_decision_file": str(DECISION_PATH),
    "sampling_strata": "year-month",
    "sample_per_year_month": SAMPLE_PER_YEAR_MONTH,
    "sampling_eligibility": (
        "at least one target horizon observed; feature NaNs retained"
    ),
    "n_rows_sample": len(sample),
    "study_start": str(sample["time"].min()),
    "study_end": str(sample["time"].max()),
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2)
)

print(f"Features ({len(FEATURE_COLS)}):")
for column in FEATURE_COLS:
    print(f"  {column}")

print(f"\nTargets ({len(TARGET_COLS)}): {TARGET_COLS}")
print(f"\nManifest → {MANIFEST_PATH}")

Features (15):
  log_chl_t0
  log_chl_lag1
  log_chl_lag2
  log_chl_lag3
  log_chl_lag7
  log_chl_lag14
  log_chl_roll7_mean
  log_chl_roll7_std
  sst_t0
  sst_lag1
  sst_roll7_mean
  depth_m
  doy_sin
  doy_cos
  gap_age

Targets (7): ['target_h1', 'target_h2', 'target_h3', 'target_h4', 'target_h5', 'target_h6', 'target_h7']

Manifest → data\processed\feature_manifest.json


---
## Next: `05-Baselines.ipynb`

Computes the **persistence** and **climatology** baselines — the bars every model in Stage 6 onwards must clear. PLAN.md: *"A fancy model that can't beat persistence is worthless, and you can only know that if you compute persistence first."*